# ☂️ Thai Parasol 3D Reconstruction: 3DGS & NeuS on Kaggle

This notebook trains **3D Gaussian Splatting (`splatfacto`)** or **NeuS (`neus-facto`)** to learn a high-fidelity 3D representation and extracts a **watertight 3D triangle mesh** and photorealistic splat.

### ⚙️ Required Kaggle Settings (Right Sidebar):
1. **Accelerator:** `GPU T4 x2` or `GPU P100` (Free 30 hrs/week)
2. **Internet:** **`On`** (Toggle on in the Notebook Options panel to allow `pip install`)
3. **Persistence:** `No persistence` or `Files only`

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi

## 📦 Step 2: Unpack and Verify Dataset
The code below searches both `/kaggle/input/` and `/kaggle/working/` for `dataset_neural.zip` (or your uploaded dataset) and unpacks it cleanly into `/kaggle/working/dataset`.

In [ ]:
import os, shutil, glob, json
from pathlib import Path

# Clean working directory for dataset
working_dataset = Path("/kaggle/working/dataset")
if working_dataset.exists():
    shutil.rmtree(working_dataset)
working_dataset.mkdir(parents=True, exist_ok=True)

# Search for dataset_neural.zip in input or working
zip_candidates = glob.glob("/kaggle/input/**/dataset_neural*.zip", recursive=True) + \
                 glob.glob("/kaggle/input/**/*.zip", recursive=True) + \
                 glob.glob("/kaggle/working/*.zip")

if zip_candidates:
    target_zip = zip_candidates[0]
    print(f"📦 Found dataset zip: {target_zip}")
    !unzip -q -o "{target_zip}" -d /kaggle/working/dataset
else:
    # Check if files were uploaded uncompressed in /kaggle/input/
    input_folders = glob.glob("/kaggle/input/**/images", recursive=True)
    if input_folders:
        src_root = Path(input_folders[0]).parent
        print(f"📦 Copying uncompressed dataset from: {src_root}")
        shutil.copytree(src_root, working_dataset, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            "❌ No dataset found! Please click 'Add Data' (or 'Upload') on the right sidebar "
            "and upload 'dataset_neural.zip'."
        )

# Handle nested root if zip extracted inside a subfolder
if not (working_dataset / "images").exists():
    for sub in working_dataset.glob("**/*"):
        if (sub / "images").exists():
            for item in sub.iterdir():
                shutil.move(str(item), str(working_dataset))
            break

# Ensure images subfolders (cv-1, cv-2, etc.) are reachable both at root and under images/
for ring in ["cv-1", "cv-2", "cv-3", "cv-4"]:
    r_in_images = working_dataset / "images" / ring
    r_in_root = working_dataset / ring
    if r_in_images.exists() and not r_in_root.exists():
        os.symlink(str(r_in_images.resolve()), str(r_in_root))

# Ensure meta_data.json exists (guarantees NeuS exporter never throws FileNotFoundError)
meta_path = working_dataset / "meta_data.json"
tf_path = working_dataset / "transforms.json"
if tf_path.exists() and not meta_path.exists():
    with open(tf_path, "r") as f:
        tf = json.load(f)
    K = [[tf["fl_x"], 0.0, tf["cx"]], [0.0, tf["fl_y"], tf["cy"]], [0.0, 0.0, 1.0]]
    sdf_frames = [{"rgb_path": fr["file_path"], "camtoworld": fr["transform_matrix"], "intrinsics": K} for fr in tf["frames"]]
    meta_dict = {
        "camera_model": "OPENCV", "height": tf["h"], "width": tf["w"],
        "has_sensor_depth": False, "has_monocular_depth": False, "has_monocular_normal": False, "has_foreground_mask": False,
        "pairs": None, "world_size": 2.0,
        "scene_box": {"aabb": [[-1.5, -1.5, -1.5], [1.5, 1.5, 1.5]], "near": 0.05, "far": 3.5, "radius": 1.5, "collider_type": "box"},
        "frames": sdf_frames
    }
    with open(meta_path, "w") as f:
        json.dump(meta_dict, f, indent=2)

num_imgs = len(glob.glob("/kaggle/working/dataset/images/**/*.jpg", recursive=True))
print(f"\n🎉 Dataset verified! Found {num_imgs} images ready.")
print(f" - transforms.json: {tf_path.exists()}")
print(f" - meta_data.json: {meta_path.exists()}")
print(f" - sparse COLMAP model: {(working_dataset / 'sparse' / '0').exists()}")

## 🛠️ Step 3: Install Nerfstudio & Dependencies

In [ ]:
# 1. Install ninja, 3D math and open3d dependencies
!pip install -q ninja jaxtyping typeguard trimesh open3d

# 2. Install Nerfstudio and gsplat
!pip install -q gsplat==1.0.0 --no-build-isolation
!pip install -q nerfstudio==1.1.4

# 3. Fix PyTorch 2.6 weights_only=True default for all ns-export commands
import site
site_pkg = site.getsitepackages()[0]
with open(f"{site_pkg}/sitecustomize.py", "w") as f:
    f.write("""
import torch
_orig = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig(*args, **kwargs)
torch.load = _safe_load
""")

!sed -i 's/torch.load(load_path, map_location="cpu")/torch.load(load_path, map_location="cpu", weights_only=False)/g' /usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py 2>/dev/null || true

print("✅ Nerfstudio and PyTorch 2.6 compatibility patch installed!")

## 🚀 Step 4: Train 3D Gaussian Splatting (`splatfacto`)
Trains 3D Gaussian Splatting in ~5–7 minutes on T4 GPU.

In [ ]:
!rm -rf /kaggle/working/outputs_3dgs

!ns-train splatfacto \
  --data /kaggle/working/dataset \
  --viewer.quit-on-train-completion True \
  --max-num-iterations 7000 \
  --output-dir /kaggle/working/outputs_3dgs

## 📐 Step 5: Export Photorealistic Splat & 3D Mesh
Exports both the 3D Gaussian Splat PLY and the 3D Poisson Triangle Mesh, packaging them into `3dgs_parasol_results.zip`.

In [ ]:
import glob, shutil, os
from IPython.display import FileLink

# 1. Ensure PyTorch 2.6 patch is active
!sed -i 's/torch.load(load_path, map_location="cpu")/torch.load(load_path, map_location="cpu", weights_only=False)/g' /usr/local/lib/python3.12/dist-packages/nerfstudio/utils/eval_utils.py 2>/dev/null || true

# 2. Locate generated config.yml
config_files = glob.glob("/kaggle/working/outputs_3dgs/**/config.yml", recursive=True)
if not config_files:
    raise FileNotFoundError("❌ Could not find config.yml in /kaggle/working/outputs_3dgs/")
config_3dgs = config_files[0]
print(f"Using trained 3DGS config: {config_3dgs}")

# 3. Export Gaussian Splat PLY (for photorealistic browser viewing)
!rm -rf /kaggle/working/export_3dgs_splat
!ns-export gaussian-splat --load-config {config_3dgs} --output-dir /kaggle/working/export_3dgs_splat

# 4. Export Poisson 3D Triangle Mesh
!rm -rf /kaggle/working/export_3dgs_mesh
!ns-export poisson --load-config {config_3dgs} --output-dir /kaggle/working/export_3dgs_mesh

# 5. Compress results into a single zip archive
output_zip = "/kaggle/working/3dgs_parasol_results"
shutil.make_archive(output_zip, "zip", "/kaggle/working/export_3dgs_splat")
!zip -u /kaggle/working/3dgs_parasol_results.zip /kaggle/working/export_3dgs_mesh/* 2>/dev/null || true

print(f"\n🎉 SUCCESS! Results saved to: {output_zip}.zip")
print(f"File size: {os.path.getsize(output_zip + '.zip') / (1024*1024):.2f} MB")
print("Click below to download your 3D results:")
FileLink("3dgs_parasol_results.zip")